In [78]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [79]:
path = "online_retail_II.xlsx"

df_2009 = pd.read_excel(path, sheet_name='Year 2009-2010')
df_2010 = pd.read_excel(path, sheet_name='Year 2010-2011')
df_raw = pd.concat([df_2009, df_2010], ignore_index=True)
df = df_raw.copy()
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [81]:
print(f"Check missing values: \n{df.isna().sum()}")

Check missing values: 
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [82]:
mixed = df.groupby('Invoice')['Customer ID'].agg(lambda x: x.isna().any() and x.notna().any())
print(mixed.sum())

0


Rows with no Customer ID (243,007, 22.8%) were dropped. Churn is defined per customer, so these rows cannot be assigned to anyone. No invoice mixes rows with and without a Customer ID, so the missing IDs cannot be recovered from the same invoice. They are likely guest checkouts, so the model covers identified customers only.

In [83]:
print(f"Check duplicated rows: \n{df.duplicated().sum()}")

Check duplicated rows: 
34335


# Invoice

In [84]:
import re
# Replace every digit with "9" and every letter with "A" so we can see the *shape* of a value (e.g. "85123A" -> "99999A") without caring
# about the digits/letters. Values with the same shape get grouped together, so we can spot which shapes are common and which are suspicious.
def number(s):
      s = re.sub(r'[0-9]', '9', str(s))
      return s

def letter(s):
      s = re.sub(r'[A-Za-z]', 'A', s)
      return s 

# InvoiceNo: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. 
# If this code starts with letter 'c', it indicates a cancellation. 
print(df['Invoice'].apply(number).value_counts())

Invoice
999999     1047871
C999999      19494
A999999          6
Name: count, dtype: int64


In [85]:
non_c = df[df['Invoice'].astype(str).str.startswith('A')]
print(non_c)

        Invoice StockCode      Description  Quantity         InvoiceDate  \
179403  A506401         B  Adjust bad debt         1 2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1 2010-07-19 11:24:00   
403472  A528059         B  Adjust bad debt         1 2010-10-20 12:04:00   
825443  A563185         B  Adjust bad debt         1 2011-08-12 14:50:00   
825444  A563186         B  Adjust bad debt         1 2011-08-12 14:51:00   
825445  A563187         B  Adjust bad debt         1 2011-08-12 14:52:00   

           Price  Customer ID         Country  
179403 -53594.36          NaN  United Kingdom  
276274 -44031.79          NaN  United Kingdom  
403472 -38925.87          NaN  United Kingdom  
825443  11062.06          NaN  United Kingdom  
825444 -11062.06          NaN  United Kingdom  
825445 -11062.06          NaN  United Kingdom  


**Invoice format check**

Invoice numbers should be 6 digits, with a C prefix for cancellations (per the dataset documentation).
The shape check found 3 formats: 999999 (normal), C999999 (19,494 cancellations) and A999999 (6 rows).

The 6 A-prefix invoices are not documented. All of them have StockCode = B, the Description "Adjust bad debt", Quantity = 1, a large price and no Customer ID. This suggests they are internal accounting adjustments, not customer purchases.
They are not cancellations either, since their Quantity is positive.

Decision: they are removed later with all rows without a Customer ID.

In [86]:
is_c = df['Invoice'].astype(str).str.startswith('C')
is_neg = df['Quantity'] < 0

print("Number of invoice starting with C and quantity is negative", df[is_c & is_neg].shape)
print("Invoice starts with C and its quantity is not negative", df[is_c &  ~is_neg])                          

Number of invoice starting with C and quantity is negative (19493, 8)
Invoice starts with C and its quantity is not negative        Invoice StockCode Description  Quantity         InvoiceDate   Price  \
76799  C496350         M      Manual         1 2010-02-01 08:24:00  373.57   

       Customer ID         Country  
76799          NaN  United Kingdom  


This inconsistent invoice also does not have customer ID, which will be removed.

In [87]:
print("Invoice does not start with C but has negative quantity \n", df[~is_c & is_neg])
print("Invoice does not start with C but has negative quantity and customer ID\n", df[~is_c & is_neg]['Customer ID'].notna().sum())

Invoice does not start with C but has negative quantity 
         Invoice StockCode   Description  Quantity         InvoiceDate  Price  \
263      489464     21733  85123a mixed       -96 2009-12-01 10:52:00    0.0   
283      489463     71477         short      -240 2009-12-01 10:52:00    0.0   
284      489467    85123A   21733 mixed      -192 2009-12-01 10:53:00    0.0   
470      489521     21646           NaN       -50 2009-12-01 11:44:00    0.0   
3114     489655     20683           NaN       -44 2009-12-01 17:26:00    0.0   
...         ...       ...           ...       ...                 ...    ...   
1060794  581210     23395         check       -26 2011-12-07 18:36:00    0.0   
1060796  581212     22578          lost     -1050 2011-12-07 18:38:00    0.0   
1060797  581213     22576         check       -30 2011-12-07 18:38:00    0.0   
1062371  581226     23090       missing      -338 2011-12-08 09:56:00    0.0   
1064386  581422     23169       smashed      -235 2011-12-08 1

3,457 non-cancellation rows have a negative Quantity. All have Price = 0 and no Customer ID, with descriptions such as "lost", "check", "smashed" and "missing", so they are stock write-offs, not customer transactions. They are removed with the rows without a Customer ID.

In [98]:
# drop invoice without customer ID
df = df[df['Customer ID'].notna()].copy()
print(df.shape)    

(824364, 9)


# Stock Code

In [89]:
df['StockCode'] = df['StockCode'].astype(str)
df['StockCode'] = df['StockCode'].str.strip()   

In [90]:
# StockCode: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.
# Description: Product (item) name. Nominal.
print(df['StockCode'].apply(number).apply(letter).value_counts())


StockCode
99999           729813
99999A           89631
AAAA              2054
A                 1289
99999AA           1182
A9                 259
AAAAAA              61
AAAA AAAAAAA        38
AAAA999             16
AAA                 16
AAAAAA9              3
AA9999               2
Name: count, dtype: int64


In [91]:
df['Description'] = df['Description'].str.strip()
print(df['Description'].value_counts())

Description
WHITE HANGING HEART T-LIGHT HOLDER     5315
REGENCY CAKESTAND 3 TIER               3777
ASSORTED COLOUR BIRD ORNAMENT          2794
JUMBO BAG RED RETROSPOT                2776
LUNCH BAG  BLACK SKULL.                2161
                                       ... 
SET 36 COLOURING PENCILS DOILEY           1
SET 12 COLOURING PENCILS DOILEY           1
ANGEL ON KNEES METAL POLE 100CM H         1
ZINC PLANT POT HOLDER                     1
F.FAIRY,CANDLE IN GLASS,LILY/VALLEY       1
Name: count, Length: 5257, dtype: int64


In [92]:
# 72801C -> ['S/4 ROSE PINK DINNER CANDLES', '4 ROSE PINK DINNER CANDLES', 'check']
# 72801D -> ['S/4 SKY BLUE DINNER CANDLES', '4 SKY BLUE DINNER CANDLES']
# 72801c -> ['S/4 ROSE PINK DINNER CANDLES', '4 ROSE PINK DINNER CANDLES']
# 72801d -> ['S/4 SKY BLUE DINNER CANDLES', '4 SKY BLUE DINNER CANDLES']

df['StockCode'] = df['StockCode'].str.upper()

In [93]:
desc_variation = df.groupby('StockCode')['Description'].nunique()
print(desc_variation.value_counts().sort_index())
multi_desc = desc_variation[desc_variation > 1].index

for code in multi_desc:
    variants = df[df['StockCode'] == code]['Description'].unique()
    print(code, '->', list(variants))

Description
1    4047
2     549
3      43
4       7
Name: count, dtype: int64
15058A -> ['BLUE WHITE SPOTS GARDEN PARASOL', 'BLUE POLKADOT GARDEN PARASOL']
15058B -> ['PINK WHITE SPOTS GARDEN PARASOL', 'PINK POLKADOT GARDEN PARASOL']
16012 -> ['FOOD/DRINK SPUNGE STICKERS', 'FOOD/DRINK SPONGE STICKERS']
16151A -> ['FLOWER DES BLUE HANDBAG/ORANG HANDL', 'FLOWERS HANDBAG blue and orange']
16156L -> ['WRAP, CAROUSEL', 'WRAP CAROUSEL']
16161U -> ['WRAP,SUKI AND FRIENDS', 'WRAP SUKI AND FRIENDS']
17107D -> ["FLOWER FAIRY,5 SUMMER B'DRAW LINERS", 'FLOWER FAIRY 5 DRAWER LINERS', 'FLOWER FAIRY 5 SUMMER DRAW LINERS']
17129F -> ['DEEP BLUE GLASS GEMS IN BAG', 'BLUE GLASS GEMS IN BAG']
18094C -> ['WHITE/BLUE RECT CERAMIC OIL BURNER', 'WHITE AND BLUE CERAMIC OIL BURNER']
20615 -> ['BLUE SPOTTY PASSPORT COVER', 'BLUE POLKADOT PASSPORT COVER']
20622 -> ['VIPPASSPORT COVER', 'VIP PASSPORT COVER']
20652 -> ['BLUE SPOTTY LUGGAGE TAG', 'BLUE POLKADOT LUGGAGE TAG']
20658 -> ['RED SPOTTY LUGGAGE TAG', 'RED

In [94]:
def pick_description(variants):
    variants = variants.dropna()
    if variants.empty:
        return None
    upper_variants = variants[variants == variants.str.upper()]
    if not upper_variants.empty:
        return max(upper_variants, key=len)
    return max(variants, key=len)  # fallback if none are fully uppercase

stockcode_to_desc = df.groupby('StockCode')['Description'].agg(pick_description)

for code in desc_variation.sort_values(ascending=False).head(100).index:
    variants = df[df['StockCode'] == code]['Description']
    print(code, '->', pick_description(variants))

22384 -> LUNCH BAG PINK RETROSPOT
22344 -> PARTY PIZZA DISH PINK+WHITE SPOT
23196 -> RETO LEAVES MAGNETIC SHOPPING LIST
20685 -> RED SPOTTY COIR DOORMAT
23236 -> STORAGE TIN VINTAGE DOILEY
22346 -> PARTY PIZZA DISH GREEN+WHITE SPOT
22345 -> PARTY PIZZA DISH BLUE+WHITE SPOT
21955 -> DOOR MAT UNION JACK GUNS AND ROSES
84997D -> PINK 3 PIECE MINI DOTS CUTLERY SET
84997C -> BLUE 3 PIECE MINI DOTS CUTLERY SET
23131 -> MISTLETOE HEART WREATH CREAM
21524 -> SPOTTY  HOME SWEET HOME DOORMAT
23126 -> DOLLCRAFT GIRL AMELIE KIT
22536 -> MAGIC DRAWING SLATE PURDEY
84997A -> GREEN 3 PIECE MINI DOTS CUTLERY SET
22937 -> BAKING MOULD CHOCOLATE CUP CAKES
22197 -> POPCORN HOLDER , SMALL
22950 -> SET OF 36 DOILIES VINTAGE CHRISTMAS
22952 -> SET 72 CAKE CASES VINTAGE CHRISTMAS
22191 -> EAU DE NIL DINER WALL CLOCK
22740 -> POLKA DOT PEN
84509C -> SET OF 4 POLKADOT PLACEMATS
23244 -> ROUND STORAGE TIN VINTAGE LEAF
22852 -> DOG BOWL, ENAMEL , CREAM COLOUR
23203 -> JUMBO BAG DOILEY PATTERNS
23209 -> LUNCH BAG

In [95]:
df['Shape'] = df['StockCode'].apply(number).apply(letter)

correct_shapes = ['99999', '99999A', '99999AA']
minor_shapes = [s for s in df['Shape'].unique() if s not in correct_shapes ]
for shape in minor_shapes:
    print(shape)
    print(df[df['Shape'] == shape]['StockCode'].value_counts())
    print()

AAAA
StockCode
POST    2019
PADS      19
CRUK      16
Name: count, dtype: int64

A
StockCode
M    1115
D     174
Name: count, dtype: int64

A9
StockCode
C2    259
Name: count, dtype: int64

AAAA AAAAAAA
StockCode
BANK CHARGES    38
Name: count, dtype: int64

AAAA999
StockCode
TEST001    15
TEST002     1
Name: count, dtype: int64

AAAAAA
StockCode
ADJUST    61
Name: count, dtype: int64

AAAAAA9
StockCode
ADJUST2    3
Name: count, dtype: int64

AA9999
StockCode
SP1002    2
Name: count, dtype: int64

AAA
StockCode
DOT    16
Name: count, dtype: int64



In [96]:
for code in ['C2', 'C3', 'GIFT']:
    print(code)
    print(df[df['StockCode'] == code]['Description'].value_counts())
    print()

C2
Description
CARRIAGE    259
Name: count, dtype: int64

C3
Series([], Name: count, dtype: int64)

GIFT
Series([], Name: count, dtype: int64)



In [97]:
admin_codes = ['POST', 'PADS', 'CRUK', 'DOT', 'BANK CHARGES', 'AMAZONFEE',
               'M', 'D', 'S', 'B', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002']

admin_rows = df[df['StockCode'].isin(admin_codes)]
print(admin_rows.groupby('StockCode')['Customer ID'].apply(lambda x: x.isna().sum()))
print()
print(admin_rows.groupby('StockCode')['Customer ID'].apply(lambda x: x.notna().sum()))

StockCode
ADJUST          0
ADJUST2         0
BANK CHARGES    0
CRUK            0
D               0
DOT             0
M               0
PADS            0
POST            0
TEST001         0
TEST002         0
Name: Customer ID, dtype: int64

StockCode
ADJUST            61
ADJUST2            3
BANK CHARGES      38
CRUK              16
D                174
DOT               16
M               1115
PADS              19
POST            2019
TEST001           15
TEST002            1
Name: Customer ID, dtype: int64
